# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Saad-Imran-Toori/flyrank-ml-internship/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

**My lane:** Lane 4 — CTR / Engagement Opportunity Scoring. I rank visible pages by how far their
click-through rate sits below what a page like them normally gets, so an editor reviews the biggest
gaps first. This notebook writes the data contract for that lane on the real warehouse, verifies
every claim with a query, builds five honest features, and demonstrates the leakage trap.


In [4]:
# ---- Setup: connect DuckDB to the gated Hugging Face warehouse ----
import duckdb, os, numpy as np, pandas as pd
from google.colab import userdata
os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")

con = duckdb.connect()
con.sql("INSTALL httpfs; LOAD httpfs;")
con.sql("CREATE SECRET hf (TYPE huggingface, PROVIDER credential_chain);")

BASE   = "hf://datasets/FlyRank/internship-warehouse"
MONTH  = "2026-03"                       # mid-panel month; NEVER the _sample (June 2026) for label logic
FACT_M = f"{BASE}/fact_content_daily_performance/month={MONTH}/data_0.parquet"
DIMC   = f"{BASE}/dim_content.parquet"
DIMCL  = f"{BASE}/dim_clients.parquet"
FLOOR  = 500                              # impressions floor: below this, one click swings CTR too much
print("Connected. Working month:", MONTH)


Connected. Working month: 2026-03


## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

**The raw table (`fact_content_daily_performance`) grain is one row per page per client per day** —
a page-day. My lane is about *pages*, not page-days, so my analysis unit is **one row = one content
page (`content_hash_id`), aggregated over one calendar month (2026-03)** of those daily records.
I sum the month's impressions and clicks per page, and take an impression-weighted average position.

**Time window.** For this contract I use a single month, 2026-03, for both the features and the
observed CTR. That keeps the contract simple and verifiable now. The capstone will split this into a
*prior* feature window and a *later* label window (a genuine past→future setup), which the single
snapshot here cannot support — I note that as a limit in Section 4.

**What I predict / rank.** The label is the page's **observed CTR** for the month
(`clicks / impressions`), a measured quantity. The lane's score is the *shortfall* between a learned
expected CTR and this observed CTR; the ranking output is a review queue ordered by that shortfall.

The code below proves the raw grain, then builds the page-month slice and shows its row count and
date span.


In [5]:
# 1a. GRAIN — is the raw fact really one row per page, per client, per day?
dupes = con.sql(f'''
  SELECT report_date, client_hash_id, content_hash_id, COUNT(*) AS c
  FROM read_parquet("{FACT_M}")
  GROUP BY 1,2,3
  HAVING COUNT(*) > 1
  LIMIT 5
''').df()
print("Duplicate page-days (expect 0 rows):", len(dupes))

# 1b. Build the page-month slice for my lane (aggregate daily -> one row per page).
con.sql(f'''
  CREATE OR REPLACE TABLE scope AS
  SELECT
    content_hash_id,
    ANY_VALUE(client_hash_id)               AS client_hash_id,
    COUNT(*)                                AS days_active,
    SUM(gsc_impressions)                    AS impressions_m,
    SUM(gsc_clicks)                         AS clicks_m,
    SUM(gsc_sum_position)                   AS sum_pos_m,
    MIN(report_date)                        AS first_day,
    MAX(report_date)                        AS last_day
  FROM read_parquet("{FACT_M}")
  WHERE gsc_data_available IS TRUE          -- only rows where search data actually exists
  GROUP BY content_hash_id
''')

# 1c. My analysis unit really is one row per page?  (expect 0 duplicate content pages)
scope_dupes = con.sql("SELECT content_hash_id, COUNT(*) c FROM scope GROUP BY 1 HAVING c>1 LIMIT 5").df()
print("Duplicate pages in my page-month slice (expect 0 rows):", len(scope_dupes))

# 1d. Row count + date span of my slice.
span = con.sql("SELECT COUNT(*) AS pages, MIN(first_day) AS span_start, MAX(last_day) AS span_end FROM scope").df()
print(span.to_string(index=False))


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Duplicate page-days (expect 0 rows): 0


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Duplicate pages in my page-month slice (expect 0 rows): 0
 pages span_start   span_end
176738 2026-03-01 2026-03-31


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

| Bucket | Fields | Note |
|---|---|---|
| **Feature** (knowable before the click) | `gsc_avg_position` / impression-weighted position, `gsc_impressions` (volume), `content_type`, `main_intent`, `word_count`, `search_volume`, `competition`, freshness (days since `content_updated_date`) | All are true about the page *before* anyone clicks, so they are safe inputs. |
| **Label / proxy** | **`ctr` = `gsc_clicks` / `gsc_impressions`** (the thing I predict); `gsc_clicks` is the numerator | The label and anything it is computed from. Never a feature. |
| **Context** | `content_hash_id`, `client_hash_id`, `keyword_hash_id`, `url_hash_id`, `report_date`, `month` | For grouping, joining, splitting, reading — never learned from. IDs live here. |
| **Excluded** (each with a why) | `gsc_clicks` as a feature → it *is* the CTR numerator (leakage); GA4/session columns (`ga4_*`, `sessions_*`, `scroll_events`) → measured after the visit, not knowable at the ranking moment; `ai_*` breakdowns → very sparse; `is_deleted` pages → not live; product/availability flags → used only to filter, not to learn | |

The code checks **availability with `IS TRUE`** (how many rows survive) and whether **missingness follows
`content_type`** — a patterned gap I must handle, not `fillna(0)` blindly.


In [6]:
# 2a. AVAILABILITY — how many page-days survive the IS TRUE filters?
avail = con.sql(f'''
  SELECT
    COUNT(*)                                                   AS all_rows,
    COUNT(*) FILTER (WHERE gsc_data_available IS TRUE)         AS gsc_available,
    COUNT(*) FILTER (WHERE ga4_data_available IS TRUE)         AS ga4_available,
    COUNT(*) FILTER (WHERE client_has_gsc     IS TRUE)         AS client_has_gsc_true
  FROM read_parquet("{FACT_M}")
''').df()
print("Availability of page-day rows in", MONTH)
print(avail.to_string(index=False))

# 2b. MISSINGNESS by content_type — is a key feature (word_count) missing along category lines?
miss = con.sql(f'''
  SELECT content_type,
         COUNT(*) AS pages,
         ROUND(AVG(CASE WHEN word_count   IS NULL THEN 1.0 ELSE 0 END)*100,1) AS pct_word_count_null,
         ROUND(AVG(CASE WHEN main_intent  IS NULL THEN 1.0 ELSE 0 END)*100,1) AS pct_intent_null,
         ROUND(AVG(CASE WHEN search_volume IS NULL THEN 1.0 ELSE 0 END)*100,1) AS pct_search_vol_null
  FROM read_parquet("{DIMC}")
  GROUP BY content_type
  ORDER BY pages DESC
''').df()
print("\nMissingness by content_type (if it varies by row, the gap is patterned, not random):")
print(miss.to_string(index=False))


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Availability of page-day rows in 2026-03
 all_rows  gsc_available  ga4_available  client_has_gsc_true
  9841378        3611061         413966              9841378

Missingness by content_type (if it varies by row, the gap is patterned, not random):
      content_type  pages  pct_word_count_null  pct_intent_null  pct_search_vol_null
   keyword article 459174                 38.1             19.9                 18.6
    feedly article  57024                  5.1            100.0                100.0
comparison article   3408                  0.1              0.1                  0.1


## 3. Verify it with queries (grain, counts, missing values, windows) — plus five features and the leakage trap

*Every claim above gets a query cell. A contract claim without a query next to it is a guess.*

Here I build the **five-feature frame** for my lane from month 2026-03, give each feature its
"knowable at the decision moment because…" line, then run the **deliberate leakage experiment**: I
add one label-derived column, watch a quick score jump toward perfect, delete it, and keep the
honest number.

**The five features (each knowable at the ranking moment):**
1. `avg_position_m` — impression-weighted average search position. *Knowable because position is set by the search engine before any click is counted.*
2. `log_impressions_m` — log of monthly impressions (volume). *Knowable because impressions are exposure, which happens before the click.*
3. `content_type` — page type. *Knowable because it is a fixed property of the page.*
4. `main_intent` — the keyword's intent. *Knowable because it is metadata assigned when the page/keyword was created.*
5. `word_count` — article length. *Knowable because it is a property of the published page, fixed before the visit.*

The label is `ctr = clicks_m / impressions_m * 100` — an observed measurement, never a feature.


In [7]:
# 3a. Build the five-feature frame (page-month join to page metadata), with the label.
frame = con.sql(f'''
  SELECT
    s.content_hash_id,
    s.client_hash_id,
    s.impressions_m,
    s.clicks_m,
    (s.sum_pos_m * 1.0 / NULLIF(s.impressions_m,0))     AS avg_position_m,
    d.content_type,
    d.main_intent,
    d.word_count
  FROM scope s
  JOIN read_parquet("{DIMC}") d USING (content_hash_id)
  WHERE s.impressions_m >= {FLOOR}          -- volume floor: CTR below this is noise
''').df()

frame["ctr"]               = frame["clicks_m"] / frame["impressions_m"] * 100.0
frame["log_impressions_m"] = np.log1p(frame["impressions_m"])
frame["content_type"]      = frame["content_type"].fillna("unknown")
frame["main_intent"]       = frame["main_intent"].fillna("unknown")
frame["word_count"]        = frame["word_count"].fillna(frame["word_count"].median())

print("Feature frame:", frame.shape[0], "pages")
print("Median CTR in slice:", round(frame["ctr"].median(),3), "%")
print(frame[["avg_position_m","log_impressions_m","content_type","main_intent","word_count","ctr"]].head())


Feature frame: 61924 pages
Median CTR in slice: 0.178 %
   avg_position_m  log_impressions_m     content_type    main_intent  \
0       12.746770           6.652863  keyword article  informational   
1        6.675782           8.174139  keyword article     commercial   
2       53.148184           7.646354  keyword article  informational   
3       10.112145           7.568379  keyword article  informational   
4        9.352863           7.727976  keyword article  informational   

   word_count       ctr  
0        3804  0.645995  
1        2853  0.592050  
2        3965  0.047801  
3        3612  0.361757  
4        2783  0.000000  


In [8]:
# 3b. THE TRAP — add ONE label-derived column and watch the score jump, then remove it.
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.tree import DecisionTreeRegressor
from sklearn.model_selection import cross_val_score

y = frame["ctr"].values
num_honest = ["avg_position_m", "log_impressions_m", "word_count"]
cat        = ["content_type", "main_intent"]

def quick_r2(num_cols):
    pre = ColumnTransformer([
        ("num", "passthrough", num_cols),
        ("cat", OneHotEncoder(handle_unknown="ignore"), cat),
    ])
    model = Pipeline([("pre", pre), ("tree", DecisionTreeRegressor(max_depth=6, random_state=0))])
    scores = cross_val_score(model, frame, y, cv=5, scoring="r2")
    return scores.mean()

honest = quick_r2(num_honest)
print(f"HONEST five features        -> mean CV R^2 = {honest:.3f}")

# Add the leak: clicks_m is the numerator of CTR, so the model can reconstruct the label.
frame["clicks_m_LEAK"] = frame["clicks_m"]
leaked = quick_r2(num_honest + ["clicks_m_LEAK"])
print(f"WITH clicks_m added (LEAK)  -> mean CV R^2 = {leaked:.3f}   <- looks perfect, but it cheated")

# Remove the leak and keep the honest number.
frame.drop(columns=["clicks_m_LEAK"], inplace=True)
print(f"\nRemoved the leak. Honest R^2 stands at {honest:.3f}.")
print("Lesson: clicks is part of the CTR label, so using it inflates the score without discovering anything.")


HONEST five features        -> mean CV R^2 = 0.037
WITH clicks_m added (LEAK)  -> mean CV R^2 = 0.874   <- looks perfect, but it cheated

Removed the leak. Honest R^2 stands at 0.037.
Lesson: clicks is part of the CTR label, so using it inflates the score without discovering anything.


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

**One named limitation of my slice:** this is a **single-month snapshot**, so I can only relate a
page's features to CTR *observed in the same window* — I cannot yet prove a page will keep
under-performing, because I have no *later* window to check against. That forward-looking label needs
the multi-month history, which I will use in the capstone.

Two more limits the code shows:
- **Unbalanced history** — clients start tracking on different dates (`gsc_data_start`), so earlier
  months are missing for some clients rather than being zero.
- **GSC-only / three-valued flags** — availability flags can be FALSE or NULL, not just TRUE, so
  filtering must use `IS TRUE`, never `= FALSE`.

All claims here are **observed, directional and decision-support only** — the CTR relationships are
observational, never causal, and I never claim to have "predicted Google."


In [9]:
# 4. Show the unbalanced panel: clients begin tracking on different dates.
hist = con.sql(f'''
  SELECT
    COUNT(*)                                                    AS clients,
    MIN(gsc_data_start)                                         AS earliest_gsc_start,
    MAX(gsc_data_start)                                         AS latest_gsc_start,
    COUNT(*) FILTER (WHERE has_gsc_access IS TRUE)              AS have_gsc,
    COUNT(*) FILTER (WHERE has_ga4_access IS TRUE)              AS have_ga4,
    COUNT(*) FILTER (WHERE has_gsc_access IS NULL)              AS gsc_flag_null
  FROM read_parquet("{DIMCL}")
''').df()
print("Client history is unbalanced — tracking starts on different dates:")
print(hist.to_string(index=False))


Client history is unbalanced — tracking starts on different dates:
 clients earliest_gsc_start latest_gsc_start  have_gsc  have_ga4  gsc_flag_null
     104         2025-01-27       2026-06-02        67        54             10


## Self-check

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere — only pseudonymous hash IDs and aggregates
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.

**Contract in one paragraph.** One row is one content page over month 2026-03, built from the daily
`fact_content_daily_performance` table joined to `dim_content`. I rank pages by the shortfall between
a learned expected CTR and the observed CTR (`clicks / impressions`); `gsc_clicks` and anything
derived from the label are excluded from the features, GA4/session columns are excluded as
after-the-fact, and IDs are context only. Availability is filtered with `IS TRUE`. The one limit that
matters most is that this single-month snapshot cannot support a past→future label yet — that is
capstone work.
